In [ ]:
from pathlib import Path

import matplotlib

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import sys
from pathlib import Path

parent_dir = str(Path.cwd().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from experiment_common import (
    FS,
    DT,
    EXPECTED_SAMPLES,
    CWT_FREQUENCIES,

    ricker_wavelet,
    add_noise,
    cwt_morlet_pywt,
    hos_preprocess_cwt,
    inverse_cwt,
    get_analysis_scales,
)


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", "./.mpl-cache")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from joblib import Parallel, delayed, parallel_config
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import MinMaxScaler

import experiment_common as ec

WAVEFORM = "ricker"
FREQUENCIES = [100,]
SNR_VALUES = list(range(-10, 7, 2))
N_REPEAT = 200
N_JOBS = 6
BASE_SEED = 20260818

WINDOW_SIZE = ec.WINDOW_SIZE
SHORT_SIZE = ec.SHORT_SIZE
LONG_SIZE = ec.LONG_SIZE

FEATURE_NAMES = ("M", "P", "SLTA", "Std", "S", "K")
OUTPUT_DIR = Path("results_feature_relevance_global_snr")
FIGURE_DIR = OUTPUT_DIR / "figures"

def build_binary_labels(clean):

    x = np.asarray(clean, dtype=float).ravel()
    onset = ec.aic_reference_arrival(x)
    reverse_onset = ec.aic_reference_arrival(x[::-1])
    end = x.size - 1 - reverse_onset

    peak = int(np.argmax(np.abs(x)))
    if not (onset <= peak <= end):
        raise RuntimeError(f"Invalid AIC label range: onset={onset}, peak={peak}, end={end}")

    labels = np.zeros(x.size, dtype=np.int8)
    labels[onset:end + 1] = 1
    return labels, int(onset), int(end)

def calculate_features(enhanced):

    features = np.column_stack([
        ec.get_amplitude(enhanced, WINDOW_SIZE),
        ec.get_energy(enhanced, WINDOW_SIZE),
        ec.get_SLTA(enhanced, SHORT_SIZE, LONG_SIZE),
        ec.get_std(enhanced, WINDOW_SIZE),
        ec.get_skewness(enhanced, WINDOW_SIZE),
        ec.get_kurtosis(enhanced, WINDOW_SIZE),
    ]).astype(float)

    np.nan_to_num(features, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return features

def safe_abs_corr(func, feature, labels):
    if np.std(feature) <= np.finfo(float).eps or np.std(labels) <= np.finfo(float).eps:
        return 0.0
    value = func(feature, labels)[0]
    return float(abs(value)) if np.isfinite(value) else 0.0

def calculate_scores(features, labels, seed):
    scaled = MinMaxScaler().fit_transform(features)

    pearson = np.array([
        safe_abs_corr(pearsonr, scaled[:, i], labels)
        for i in range(scaled.shape[1])
    ])

    spearman = np.array([
        safe_abs_corr(spearmanr, scaled[:, i], labels)
        for i in range(scaled.shape[1])
    ])

    mi = mutual_info_classif(
        scaled, labels, discrete_features=False,
        n_neighbors=3, random_state=seed
    )
    mi = np.nan_to_num(mi, nan=0.0, posinf=0.0, neginf=0.0)
    return pearson, spearman, mi

def run_one_condition(frequency_index, frequency, snr_db, labels, label_onset, label_end):
    rows = []

    for repeat in range(N_REPEAT):
        seed_seq = np.random.SeedSequence([BASE_SEED, frequency_index, repeat])
        seed = int(seed_seq.generate_state(1, dtype=np.uint32)[0])
        rng = np.random.default_rng(seed)

        noisy, _, metadata = ec.simulate(
            WAVEFORM, frequency, snr_db, noise_type="WGN", rng=rng
        )
        enhanced, kept_idx, _ = ec.cwt_hos_icwt(noisy)
        features = calculate_features(enhanced)
        pearson, spearman, mi = calculate_scores(features, labels, seed)

        for i, feature in enumerate(FEATURE_NAMES):
            rows.append({
                "frequency_hz": int(frequency),
                "snr_db": int(snr_db),
                "repeat": int(repeat),
                "feature": feature,
                "window_size": int(WINDOW_SIZE),
                "sta_size": int(SHORT_SIZE),
                "lta_size": int(LONG_SIZE),
                "label_onset": int(label_onset),
                "label_end": int(label_end),
                "label_width_samples": int(label_end - label_onset + 1),
                "retained_scale_count": int(kept_idx.size),
                "snr_definition": metadata["snr_definition"],
                "pearson_abs": float(pearson[i]),
                "spearman_abs": float(spearman[i]),
                "mutual_information": float(mi[i]),
            })

    return rows

def summarize_frequency_snr(raw):
    return raw.groupby(
        ["frequency_hz", "snr_db", "feature"], as_index=False, observed=True
    ).agg(
        n_records=("repeat", "count"),
        pearson_mean=("pearson_abs", "mean"),
        pearson_std=("pearson_abs", "std"),
        spearman_mean=("spearman_abs", "mean"),
        spearman_std=("spearman_abs", "std"),
        mi_mean=("mutual_information", "mean"),
        mi_std=("mutual_information", "std"),
        mean_retained_scale_count=("retained_scale_count", "mean"),
    )

def summarize_snr(summary_fs):

    return summary_fs.groupby(
        ["snr_db", "feature"], as_index=False, observed=True
    ).agg(
        frequency_count=("frequency_hz", "nunique"),
        pearson_mean=("pearson_mean", "mean"),
        pearson_frequency_std=("pearson_mean", "std"),
        spearman_mean=("spearman_mean", "mean"),
        spearman_frequency_std=("spearman_mean", "std"),
        mi_mean=("mi_mean", "mean"),
        mi_frequency_std=("mi_mean", "std"),
    )

def summarize_overall(summary_snr):
    return summary_snr.groupby("feature", as_index=False, observed=True).agg(
        pearson_mean=("pearson_mean", "mean"),
        spearman_mean=("spearman_mean", "mean"),
        mi_mean=("mi_mean", "mean"),
    )

def export_metric_table(summary_snr, value_column, file_name):
    table = summary_snr.pivot(index="snr_db", columns="feature", values=value_column)
    table = table.reindex(columns=FEATURE_NAMES).reset_index()
    table.to_csv(OUTPUT_DIR / file_name, index=False, encoding="utf-8-sig")

def plot_relevance(summary_snr, save_path):
    metrics = [
        ("pearson_mean", "Mean |Pearson|"),
        ("spearman_mean", "Mean |Spearman|"),
        ("mi_mean", "Mean Mutual Information"),
    ]
    markers = ["o", "s", "^", "D", "v", "P"]

    fig, axes = plt.subplots(3, 1, figsize=(8.5, 11), sharex=True, constrained_layout=True)

    for ax, (metric, ylabel) in zip(axes, metrics):
        for feature, marker in zip(FEATURE_NAMES, markers):
            part = summary_snr[summary_snr["feature"] == feature].sort_values("snr_db")
            ax.plot(part["snr_db"], part[metric], marker=marker, lw=1.4, ms=5, label=feature)

        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)

    axes[0].legend(ncol=3, fontsize=9)
    axes[-1].set_xlabel("Global SNR (dB)")
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    label_cache = {}
    for frequency in FREQUENCIES:
        clean, _ = ec.embed_wavelet(WAVEFORM, frequency)
        labels, onset, end = build_binary_labels(clean)
        label_cache[frequency] = (labels, onset, end)

    print("=" * 72)
    print("Feature relevance: CWT + calibrated-HOS + iCWT")
    print("=" * 72)
    print(f"Frequencies: {FREQUENCIES} Hz")
    print(f"SNR: {SNR_VALUES} dB")
    print(f"Repeats: {N_REPEAT}")
    print(f"Feature window: {WINDOW_SIZE}")
    print(f"STA/LTA: {SHORT_SIZE}/{LONG_SIZE}")
    print(f"Parallel jobs: {N_JOBS}")
    print(f"SNR definition: full_trace_mean_square")
    print("-" * 72)

    for frequency in FREQUENCIES:
        _, onset, end = label_cache[frequency]
        print(f"{frequency:3d} Hz: AIC label {onset}-{end}, width={end - onset + 1} samples")

    tasks = []
    for frequency_index, frequency in enumerate(FREQUENCIES):
        labels, onset, end = label_cache[frequency]
        for snr_db in SNR_VALUES:
            tasks.append((frequency_index, frequency, snr_db, labels, onset, end))

    with parallel_config(backend="loky", inner_max_num_threads=1):
        results = Parallel(
            n_jobs=N_JOBS,
            verbose=10,
            batch_size=1,
            pre_dispatch="2*n_jobs",
        )(
            delayed(run_one_condition)(*task)
            for task in tasks
        )

    raw = pd.DataFrame([row for block in results for row in block])
    summary_fs = summarize_frequency_snr(raw)
    summary_snr = summarize_snr(summary_fs)
    overall = summarize_overall(summary_snr)

    raw.to_csv(
        OUTPUT_DIR / "feature_relevance_raw.csv",
        index=False, encoding="utf-8-sig"
    )
    summary_fs.to_csv(
        OUTPUT_DIR / "feature_summary_by_frequency_snr.csv",
        index=False, encoding="utf-8-sig"
    )
    summary_snr.to_csv(
        OUTPUT_DIR / "feature_summary_by_snr.csv",
        index=False, encoding="utf-8-sig"
    )
    overall.to_csv(
        OUTPUT_DIR / "feature_overall_summary.csv",
        index=False, encoding="utf-8-sig"
    )

    export_metric_table(summary_snr, "pearson_mean", "pearson_by_snr.csv")
    export_metric_table(summary_snr, "spearman_mean", "spearman_by_snr.csv")
    export_metric_table(summary_snr, "mi_mean", "mutual_information_by_snr.csv")
    plot_relevance(summary_snr, FIGURE_DIR / "feature_relevance_vs_snr.png")

    print("\nOverall feature relevance")
    print(overall.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print(f"\nResults saved to: {OUTPUT_DIR.resolve()}")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

path = './results_feature_relevance_global_snr/'

df_mi = pd.read_csv(path + 'mutual_information_by_snr.csv')
df_pearson = pd.read_csv(path + 'pearson_by_snr.csv')
df_spearman = pd.read_csv(path + 'spearman_by_snr.csv')

features = ['M', 'P', 'SLTA', 'Std', 'S', 'K']

styles = [
    {'linestyle': '-', 'marker': 'o', 'markersize': 6},
    {'linestyle': '--', 'marker': 's', 'markersize': 6},
    {'linestyle': '-.', 'marker': '^', 'markersize': 6},
    {'linestyle': ':', 'marker': 'D', 'markersize': 6},
    {'linestyle': '-', 'marker': 'v', 'markersize': 6},
    {'linestyle': '--', 'marker': '*', 'markersize': 8}
]
colors = plt.cm.tab10(np.linspace(0, 1, 6))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True)
data_list = [df_mi, df_pearson, df_spearman]
titles = ['Mutual Information', 'Pearson Correlation', 'Spearman Correlation']

for ax, data, title in zip(axes, data_list, titles):
    snr = data['snr_db']
    for i, feat in enumerate(features):
        ax.plot(snr, data[feat], label=feat, color=colors[i],
                linewidth=2.5, **styles[i])
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('SNR (dB)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), borderaxespad=0)

plt.tight_layout()
plt.show()
